# TickNet-L giữa kỳ — Mid224 — baseline

Huấn luyện riêng baseline TickNet-L v1 trên Mid224 theo hai đợt 1–100 và 101–200.

## Input Kaggle

Upload ZIP của project này thành Kaggle Dataset. Kaggle sẽ mount cây đã giải nén dưới `/kaggle/input/<dataset>/TickNets`, giống ảnh bạn cung cấp. Notebook tự tìm project có `train_mid_experiment.py` và `configs/midterm/*.json`; nếu có nhiều bản, đặt `PROJECT_INPUT` về đường dẫn chính xác.

Project upload cần chứa `data/Mid224/train|test/{bird,cat,dog,frog,horse}`. Không cần bật Internet và notebook không clone GitHub.

## Trước khi Run All

1. Bật **GPU**.
2. Attach Kaggle Dataset chứa project `TickNets`.
3. Phiên đầu giữ `RESUME_ARCHIVE = None`.
4. Phiên sau attach output trước đó và điền đường dẫn file ZIP hoặc thư mục Kaggle đã giải nén dưới `/kaggle/input`.

Phiên đầu tự dừng ở epoch 100. Phiên sau restore từ ZIP hoặc thư mục đã giải nén rồi tự chạy epoch 101–200; tham số `--epochs` luôn giữ 200 nên cosine scheduler không bị reset.

Source huấn luyện được đối chiếu với commit kỹ thuật `a9fb3877520aa5c28473071bec031ce9d731e52c` khi project upload có thư mục `.git`. Mỗi subprocess chạy tối đa 25 epoch rồi cập nhật summary và ZIP; notebook không tự đánh giá test.


In [ ]:
from pathlib import Path, PurePosixPath
import subprocess, sys

INPUT_ROOT = Path("/kaggle/input")
PROJECT_INPUT = None  # ví dụ: "/kaggle/input/dl-smoothing-experiment/TickNets"
REVISION = "a9fb3877520aa5c28473071bec031ce9d731e52c"
DATA = Path("/kaggle/working/mid_experiment_data")
RUN_ID = "mid224_baseline"
RUNS = Path("/kaggle/working") / f"{RUN_ID}_runs"
VARIANT = "Mid224"
CONFIGS = ["configs/midterm/baseline.json"]
EPOCHS = 200
PHASE_END_EPOCHS = (100, 200)
CHUNK_EPOCHS = 25
SESSION_BUDGET_HOURS = 10.5
NUM_WORKERS = 2
SEED = 42
RUN_PREFLIGHT_TESTS = True

# Phiên đầu để None. Phiên sau: attach output trước rồi điền đường dẫn file ZIP hoặc thư mục Kaggle đã giải nén.
RESUME_ARCHIVE = None  # ví dụ: "/kaggle/input/<previous-output>/mid224_baseline_runs_artifacts"


def locate_project_root(input_root, explicit=None):
    required = (
        Path("train_mid_experiment.py"),
        Path("configs/midterm/baseline.json"),
        Path("configs/midterm/experiment.json"),
    )

    def valid(root):
        return root.is_dir() and all((root / relative).is_file() for relative in required)

    if explicit is not None:
        root = Path(explicit)
        if not valid(root):
            raise FileNotFoundError(f"PROJECT_INPUT is not a valid TickNets project: {root}")
        return root

    candidates = sorted({
        path.parent
        for path in Path(input_root).rglob("train_mid_experiment.py")
        if valid(path.parent)
    })
    if len(candidates) != 1:
        raise ValueError(
            "Expected exactly one uploaded TickNets project under /kaggle/input; "
            f"found {candidates}. Set PROJECT_INPUT explicitly."
        )
    return candidates[0]


def phase_target_epoch(current_epoch):
    if not isinstance(current_epoch, int) or not 0 <= current_epoch <= EPOCHS:
        raise ValueError(f"Invalid current epoch: {current_epoch!r}")
    for phase_end in PHASE_END_EPOCHS:
        if current_epoch < phase_end:
            return phase_end
    return EPOCHS


def verify_uploaded_repository(repo, revision, *, runner=subprocess.run,
                               check_output=subprocess.check_output):
    runner([
        "git", "config", "--global", "--add",
        "safe.directory", str(repo),
    ], check=True)
    actual_revision = check_output(
        ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
    ).strip()
    runner([
        "git", "-C", str(repo), "diff", "--exit-code",
        "--ignore-space-at-eol", revision, "--",
        "train_mid_experiment.py",
        "models",
        "configs/midterm/baseline.json",
        "configs/midterm/experiment.json",
    ], check=True)
    return actual_revision


def build_training_command(*, python, config, variant, data_root, output, epochs,
                           seed, num_workers, current_epoch, next_epoch):
    if not 0 <= current_epoch < next_epoch <= epochs:
        raise ValueError("Expected 0 <= current_epoch < next_epoch <= epochs")
    command = [
        str(python), "train_mid_experiment.py",
        "--config", str(config),
        "--variant", variant,
        "--data-root", str(data_root),
        "--output-dir", str(output),
        "--epochs", str(epochs),
        "--seed", str(seed),
        "--num-workers", str(num_workers),
        "--device", "cuda:0",
        "--stop-after-epoch", str(next_epoch),
    ]
    if current_epoch:
        command += ["--resume", str(output / "last.pt")]
    return command


def run_training_chunk(*, python, repo, config, variant, data_root, output, epochs,
                       seed, num_workers, current_epoch, next_epoch, runner,
                       archive_callback):
    command = build_training_command(
        python=python, config=config, variant=variant, data_root=data_root,
        output=output, epochs=epochs, seed=seed, num_workers=num_workers,
        current_epoch=current_epoch, next_epoch=next_epoch,
    )
    runner(command, cwd=repo, check=True)
    return archive_callback()


In [ ]:
import csv, hashlib, json, shutil, time, zipfile

REPO = locate_project_root(INPUT_ROOT, PROJECT_INPUT)
print("Uploaded project:", REPO)

if (REPO / ".git").is_dir():
    actual_revision = verify_uploaded_repository(REPO, REVISION)
    print("Git HEAD:", actual_revision, "| training source matches:", REVISION)
else:
    actual_revision = None
    print("WARNING: uploaded project has no .git; trainer source hashes will still guard resume")

import torch
import torchvision
from torch.utils.flop_counter import FlopCounterMode  # noqa: F401 — compatibility preflight

assert sys.version_info >= (3, 10), sys.version
assert torch.cuda.is_available(), "Bật GPU trong Kaggle Notebook settings trước khi train"
print(
    "Python:", sys.version.split()[0],
    "torch:", torch.__version__,
    "torchvision:", torchvision.__version__,
    "GPU:", torch.cuda.get_device_name(0),
)
subprocess.run(["nvidia-smi"], check=True)

if RUN_PREFLIGHT_TESTS:
    subprocess.run([
        sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider",
        "tests/test_ticknet_l_stage4.py",
    ], cwd=REPO, check=True)


In [ ]:
from PIL import Image

CLASSES = ("bird", "cat", "dog", "frog", "horse")
EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff", ".ppm", ".pgm"}


def restore_runs_archive(archive_path):
    if archive_path is None:
        return False
    resume_source = Path(archive_path)
    if not resume_source.exists():
        raise FileNotFoundError(f"Resume source not found: {resume_source}")
    if RUNS.exists():
        raise FileExistsError(
            f"{RUNS} already exists; do not combine Files persistence with RESUME_ARCHIVE"
        )
    if resume_source.is_dir():
        source_runs = resume_source if resume_source.name == RUNS.name else resume_source / RUNS.name
        if not source_runs.is_dir():
            raise ValueError(
                f"Resume directory must be {RUNS.name} or contain {RUNS.name}/: {resume_source}"
            )
        shutil.copytree(source_runs, RUNS)
    else:
        with zipfile.ZipFile(resume_source) as archive:
            members = [PurePosixPath(info.filename) for info in archive.infolist()]
            if not members:
                raise ValueError("Resume archive is empty")
            if any(member.is_absolute() or ".." in member.parts for member in members):
                raise ValueError("Unsafe path in resume archive")
            if any(not member.parts or member.parts[0] != RUNS.name for member in members):
                raise ValueError(f"Archive must contain only the {RUNS.name}/ tree")
            archive.extractall(RUNS.parent)
    if not RUNS.is_dir():
        raise ValueError(f"Archive did not restore {RUNS}")
    print("Restored:", resume_source, "->", RUNS)
    return True


def has_variant_layout(root, variant):
    return all(
        (root / variant / split / class_name).is_dir()
        for split in ("train", "test")
        for class_name in CLASSES
    )


def locate_variant(variant):
    project_data = REPO / "data"
    if has_variant_layout(project_data, variant):
        return project_data / variant

    candidates = []
    for train_dir in INPUT_ROOT.rglob("train"):
        root = train_dir.parent
        if root.name.lower() != variant.lower():
            continue
        if all((root / split / class_name).is_dir()
               for split in ("train", "test") for class_name in CLASSES):
            candidates.append(root)
    if len(candidates) != 1:
        raise ValueError(
            f"Expected {REPO / 'data' / variant} or exactly one {variant} dataset "
            f"under /kaggle/input; found {candidates}"
        )
    return candidates[0]


restore_runs_archive(RESUME_ARCHIVE)
source = locate_variant(VARIANT)
expected_size = 32 if VARIANT == "Mid32" else 224
rows = []
keys = {}
for split, expected_per_class in (("train", 5000), ("test", 50)):
    keys[split] = []
    for class_name in CLASSES:
        images = sorted(
            path for path in (source / split / class_name).iterdir()
            if path.suffix.lower() in EXTENSIONS
        )
        assert len(images) == expected_per_class, (
            VARIANT, split, class_name, len(images), expected_per_class
        )
        with Image.open(images[0]) as image:
            assert image.size == (expected_size, expected_size), (images[0], image.size)
        for image_path in images:
            key = f"{class_name}/{image_path.name}"
            keys[split].append(key)
            rows.append({
                "variant": VARIANT, "split": split,
                "class_name": class_name, "filename": image_path.name,
            })
assert not set(keys["train"]).intersection(keys["test"])

DATA.mkdir(exist_ok=True)
link = DATA / VARIANT
if link.exists() or link.is_symlink():
    assert link.resolve() == source.resolve(), "Dataset link differs; restart a clean session"
else:
    link.symlink_to(source, target_is_directory=True)

manifest = DATA / "split_manifest.csv"
with manifest.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["variant", "split", "class_name", "filename"]
    )
    writer.writeheader()
    writer.writerows(rows)

RUNS.mkdir(exist_ok=True)
archived_manifest = RUNS / "source_split_manifest.csv"
if archived_manifest.is_file():
    assert archived_manifest.read_bytes() == manifest.read_bytes(), "Source manifest changed"
else:
    unexpected_runs = [
        RUNS / f"{VARIANT.lower()}_{Path(config).stem}_seed{SEED}"
        for config in CONFIGS
        if (RUNS / f"{VARIANT.lower()}_{Path(config).stem}_seed{SEED}").exists()
    ]
    assert not unexpected_runs, f"Run directories exist without provenance: {unexpected_runs}"
    archived_manifest.write_bytes(manifest.read_bytes())

revision_file = RUNS / "source_revision.txt"
if revision_file.is_file():
    assert revision_file.read_text(encoding="utf-8").strip() == REVISION
else:
    revision_file.write_text(REVISION + "\n", encoding="utf-8")

print(
    "Data:", source,
    "samples:", len(rows),
    "manifest SHA256:", hashlib.sha256(manifest.read_bytes()).hexdigest(),
)


In [ ]:
def checkpoint_epoch(output):
    if not output.exists():
        return 0
    checkpoint_path = output / "last.pt"
    if not checkpoint_path.is_file():
        raise FileNotFoundError(
            f"{output} exists without last.pt; inspect or remove this incomplete run directory"
        )
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    if checkpoint.get("role") != "last":
        raise ValueError(f"{checkpoint_path} is not a resumable last checkpoint")
    epoch = checkpoint.get("epoch")
    if not isinstance(epoch, int) or not 1 <= epoch <= EPOCHS:
        raise ValueError(f"Invalid checkpoint epoch in {checkpoint_path}: {epoch!r}")
    return epoch


def write_artifacts():
    summary = []
    for config_path in CONFIGS:
        label = Path(config_path).stem
        run = RUNS / f"{VARIANT.lower()}_{label}_seed{SEED}"
        epoch_log = run / "epochs.csv"
        config_log = run / "config.json"
        if not epoch_log.is_file() or not config_log.is_file():
            continue
        config = json.loads(config_log.read_text(encoding="utf-8"))
        with epoch_log.open(encoding="utf-8", newline="") as handle:
            epochs = list(csv.DictReader(handle))
        if not epochs:
            continue
        valid = [row for row in epochs if row["val_top1"]]
        best = max(
            valid,
            key=lambda row: (float(row["val_top1"]), -float(row["val_loss"])),
        ) if valid else epochs[-1]
        summary.append({
            "run": run.name,
            "variant": config["variant"],
            "seed": config["seed"],
            "revision": config["git_revision"],
            "model": config["model"],
            "completed_epoch": int(epochs[-1]["epoch"]),
            "target_epochs": EPOCHS,
            "best_epoch": int(best["epoch"]),
            "val_top1": best["val_top1"],
            "val_loss": best["val_loss"],
            "parameters": config["complexity"]["learnable_parameters"],
            "gflops": config["complexity"]["flops"] / 1e9,
            "validation_sha256": config["validation_sha256"],
        })
    summary_path = RUNS / "validation_summary.csv"
    if summary:
        with summary_path.open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(summary[0]))
            writer.writeheader()
            writer.writerows(summary)
        for row in summary:
            print(row)
    archive_path = shutil.make_archive(
        str(RUNS) + "_artifacts",
        "zip",
        root_dir=RUNS.parent,
        base_dir=RUNS.name,
    )
    print("Checkpoint artifact refreshed:", archive_path)
    return Path(archive_path)


jobs = []
for config_path in CONFIGS:
    label = Path(config_path).stem
    jobs.append((config_path, RUNS / f"{VARIANT.lower()}_{label}_seed{SEED}"))

session_targets = {
    output.name: phase_target_epoch(checkpoint_epoch(output))
    for _, output in jobs
}
print("This Kaggle session targets:", session_targets)

session_started = time.monotonic()
budget_seconds = SESSION_BUDGET_HOURS * 3600
estimated_chunk_seconds = {}
archive_path = write_artifacts()
paused_for_budget = False

while True:
    pending = False
    made_progress = False
    for config_path, output in jobs:
        current_epoch = checkpoint_epoch(output)
        target_epoch = session_targets[output.name]
        if current_epoch >= target_epoch:
            print("Phase complete:", output.name, current_epoch, "/", target_epoch)
            continue
        pending = True

        elapsed = time.monotonic() - session_started
        fallback_estimate = max(estimated_chunk_seconds.values(), default=0.0)
        estimate = estimated_chunk_seconds.get(output.name, fallback_estimate)
        if elapsed >= budget_seconds or (
            estimate and elapsed + 1.25 * estimate > budget_seconds
        ):
            paused_for_budget = True
            break

        next_epoch = min(current_epoch + CHUNK_EPOCHS, target_epoch)
        print("Training:", output.name, f"epoch {current_epoch + 1}..{next_epoch}/{EPOCHS}")
        chunk_started = time.monotonic()
        archive_path = run_training_chunk(
            python=sys.executable,
            repo=REPO,
            config=config_path,
            variant=VARIANT,
            data_root=DATA,
            output=output,
            epochs=EPOCHS,
            seed=SEED,
            num_workers=NUM_WORKERS,
            current_epoch=current_epoch,
            next_epoch=next_epoch,
            runner=subprocess.run,
            archive_callback=write_artifacts,
        )
        estimated_chunk_seconds[output.name] = time.monotonic() - chunk_started
        made_progress = True

    if paused_for_budget or not pending:
        break
    if not made_progress:
        raise RuntimeError("Training scheduler made no progress")

progress = {output.name: checkpoint_epoch(output) for _, output in jobs}
unfinished_total = {name: epoch for name, epoch in progress.items() if epoch < EPOCHS}
phase_complete = all(
    progress[name] >= target for name, target in session_targets.items()
)
if paused_for_budget:
    print("Paused cleanly before Kaggle's 12-hour limit:", progress)
elif phase_complete:
    print("Planned phase complete:", progress)
if unfinished_total:
    print("Attach this ZIP to the next session and set RESUME_ARCHIVE:", archive_path)
else:
    print("All 200 epochs completed:", archive_path)


## Sau khi train

Artifact nằm tại `/kaggle/working/mid224_baseline_runs_artifacts.zip`.

Nếu chưa đủ 200 epoch, lưu Kaggle output, attach vào phiên tiếp theo và đặt `RESUME_ARCHIVE` về file ZIP hoặc thư mục Kaggle đã giải nén trong `/kaggle/input`. Trainer tiếp tục optimizer, cosine scheduler và RNG từ `last.pt`.

Chỉ đánh giá test sau khi chốt cấu hình bằng validation. Dùng `--evaluate <RUN>/best_val.pt` với output directory mới; không dùng test để chọn model, seed, LR hoặc epoch.
